# ExPECA end-to-end OAI 5G: one gNB and two SDR UEs

This notebook provisions the verified three-worker/three-SDR topology, starts the OAI 5G Core and gNB, attaches two software UEs, installs the UE routes, runs D-ITG smoke traffic, and checks the RAN, AMF, UPF, and user plane.

Run the cells from top to bottom. They are intentionally idempotent where practical: active leases and containers are reused, while OAI radio processes are restarted cleanly. The notebook never releases leases or deletes containers automatically.

## Verified topology

| Role | Worker | SDR | Container SDR address | UHD device address | IMSI |
|---|---|---|---|---|---|
| gNB + 5GC | `worker-01` | `sdr-03` | `10.30.10.121/24` | `10.30.10.6` | — |
| UE 1 | `worker-06` | `sdr-05` | `10.30.10.122/24` | `10.30.10.10` | `001010000000001` |
| UE 2 | `worker-04` | `sdr-04` | `10.30.10.123/24` | `10.30.10.8` | `001010000000002` |

`sdr-01` is deliberately not used: it could synchronize downlink but did not sustain a reliable uplink in the bring-up. `sdr-04` attached UE 2 immediately. The start order below is core → gNB → UE 2 → UE 1; this was the reliable two-UE sequence.

## 1. Inputs

Set the ExPECA project and OpenRC path. The extra worker remains a top-cell input, but its verified default is `worker-04`. Do not place your ExPECA password in the notebook.

In [ ]:
project = "YOUR-EXPECA-PROJECT"
openrc_path = f"/content/{project}-openrc.sh"  # change for local Jupyter
lease_days = 7

# The third worker is the requested input parameter.
extra_worker_name = "worker-04"

radio_hosts = [
    {
        "role": "gnb", "container": "oai-gnb",
        "worker": "worker-01", "sdr": "sdr-03",
        "sdr_host_cidr": "10.30.10.121/24", "sdr_device_ip": "10.30.10.6",
    },
    {
        "role": "ue1", "container": "oai-ue1",
        "worker": "worker-06", "sdr": "sdr-05",
        "sdr_host_cidr": "10.30.10.122/24", "sdr_device_ip": "10.30.10.10",
        "imsi": "001010000000001",
    },
    {
        "role": "ue2", "container": "oai-ue2",
        "worker": extra_worker_name, "sdr": "sdr-04",
        "sdr_host_cidr": "10.30.10.123/24", "sdr_device_ip": "10.30.10.8",
        "imsi": "001010000000002",
    },
]

container_image = "samiemostafavi/sshd-dind-oai"
container_password = "expeca"  # password inside the disposable lab containers
public_gateway = "130.237.11.97"
dns_ip = "8.8.8.8"

oai_root = "/root/openairinterface5g-edaf"
oai_build = f"{oai_root}/cmake_targets/ran_build/build"
gnb_config = f"{oai_root}/targets/PROJECTS/GENERIC-NR-5GC/CONF/gnb.sa.band77.fr1.106PRB.usrpb210.conf"
core_root = "/root/oai-cn5g"
gnb_userplane_ip = "192.168.70.129"
core_addresses = {"amf": "192.168.70.132", "smf": "192.168.70.133", "upf": "192.168.70.134"}
ue_gateway = "10.0.0.1"

ue_rxgain = 115
ue_txgain = 0
ue_attach_attempts = 3
ue_attach_timeout_s = 180
ditg_idt_ms = 50.0
ditg_packet_bytes = 100
ditg_duration_ms = 10_000

## 2. Authentication and dependencies

Download the project OpenRC file from the ExPECA API access page. The next cells install `python-chi`, import the OpenRC variables, and prompt for the password without storing or displaying it.

In [ ]:
%pip uninstall -q -y moviepy
%pip install -q jedi git+https://github.com/KTH-EXPECA/python-chi

In [ ]:
import base64
import ipaddress
import json
import os
import re
import shlex
import time
from getpass import getpass
from pathlib import Path

if project == "YOUR-EXPECA-PROJECT":
    raise ValueError("Set project in the input cell first.")

openrc_text = Path(openrc_path).read_text()
for name, value in re.findall(r'export\s+(\w+)\s*=\s*("[^"]+"|[^"\n]+)', openrc_text):
    os.environ[name] = value.strip().strip('\"')
os.environ["OS_PASSWORD"] = getpass("ExPECA password: ")

from loguru import logger
import chi.container
import chi.network
from chi.expeca import (
    get_available_publicips, get_container_status, get_radio_interfaces,
    get_segment_ids, get_worker_interfaces, list_reservations, reserve,
    show_reservation_byname,
)

## 3. Execution helpers and safety checks

Commands are sent through the command service already present in the container image. Base64 transport avoids shell-quoting errors. Every foreground command receives an internal return-code marker; long-lived OAI and D-ITG processes are launched with `setsid -f`.

In [ ]:
def validate_topology():
    if [h["role"] for h in radio_hosts] != ["gnb", "ue1", "ue2"]:
        raise ValueError("radio_hosts must be ordered gnb, ue1, ue2.")
    for field in ("container", "worker", "sdr", "sdr_host_cidr", "sdr_device_ip"):
        values = [h[field] for h in radio_hosts]
        if len(values) != len(set(values)):
            raise ValueError(f"Duplicate {field}: {values}")
    for h in radio_hosts:
        ipaddress.ip_interface(h["sdr_host_cidr"])
        ipaddress.ip_address(h["sdr_device_ip"])
    if any(h["sdr"] == "sdr-01" for h in radio_hosts):
        logger.warning("sdr-01 was uplink-unstable in the verified run; use it only for diagnosis.")

def _command_output(result):
    output = result.get("output", "") if isinstance(result, dict) else str(result)
    try:
        decoded = json.loads(output)
        if isinstance(decoded, dict) and "output" in decoded:
            output = decoded["output"]
    except (TypeError, json.JSONDecodeError):
        pass
    return str(output)

def remote(container, command, check=True):
    marked = (
        "set -o pipefail\n" + command +
        "\nrc=$?\nprintf '\\n__REMOTE_RC__=%s\\n' \"$rc\"\nexit 0\n"
    )
    encoded = base64.b64encode(marked.encode()).decode()
    inner = f"echo {shlex.quote(encoded)} | base64 -d | bash"
    payload = json.dumps({"cmd": inner})
    result = chi.container.execute(
        container_ref=container,
        command=(
            "curl -sS -X POST -H 'Content-Type: application/json' "
            f"--data-binary {shlex.quote(payload)} http://localhost:50505/"
        ),
    )
    output = _command_output(result)
    match = re.search(r"__REMOTE_RC__=(\d+)", output)
    rc = int(match.group(1)) if match else None
    clean = re.sub(r"\n?__REMOTE_RC__=\d+\n?", "", output).strip()
    if check and rc not in (0,):
        raise RuntimeError(f"{container}: remote command failed (rc={rc})\n{clean}")
    return clean, rc

def wait_until(description, predicate, timeout_s, interval_s=5):
    deadline = time.monotonic() + timeout_s
    last = None
    while time.monotonic() < deadline:
        try:
            last = predicate()
            if last:
                logger.success(description)
                return last
        except Exception as exc:
            last = exc
        time.sleep(interval_s)
    raise TimeoutError(f"Timed out waiting for {description}; last result: {last}")

def tunnel_cidr(container):
    out, _ = remote(container, "ip -4 -o addr show dev oaitun_ue1 2>/dev/null || true", check=False)
    match = re.search(r"\binet\s+(\d+(?:\.\d+){3}/\d+)", out)
    return match.group(1) if match else None

validate_topology()
logger.success("Topology validated.")

## 4. Inspect and reserve the three SDR/worker pairs

Both SDR ports are reserved because the management and sample paths are separate ExPECA networks. Existing leases with the same names are reused. The cell waits until each worker lease is active before provisioning.

In [ ]:
for host in radio_hosts:
    status = get_radio_interfaces(host["sdr"])
    logger.info(f"{host['role']} / {host['sdr']}:\n{json.dumps(status, indent=2)}")
    down = [port for port, details in status.items() if details.get("linkstate") != "Up"]
    if down:
        raise RuntimeError(f"Required ports are down on {host['sdr']}: {down}")

def existing_or_new_lease(lease_name, spec):
    lease = show_reservation_byname(lease_name)
    return lease if lease else reserve(spec)

def reserve_host(host):
    segments = get_segment_ids(host["sdr"])
    for port in ("rj45", "sfp"):
        name = f"{host['sdr']}-{port}-lease"
        existing_or_new_lease(name, {
            "type": "network", "name": f"{host['sdr']}-{port}",
            "net_name": f"{host['sdr']}-{port}", "segment_id": segments[port],
            "duration": {"days": lease_days, "hours": 0},
        })
    worker_name = f"{host['worker']}-lease"
    lease = existing_or_new_lease(worker_name, {
        "type": "device", "name": host["worker"],
        "duration": {"days": lease_days, "hours": 0},
    })
    host["worker_lease_name"] = worker_name
    host["worker_reservation_id"] = lease["reservations"][0]["id"]

for host in radio_hosts:
    reserve_host(host)

def worker_lease_active(host):
    lease = show_reservation_byname(host["worker_lease_name"]) or {}
    return str(lease.get("status", "")).upper() == "ACTIVE"

for host in radio_hosts:
    wait_until(
        f"{host['worker']} lease is ACTIVE",
        lambda h=host: worker_lease_active(h), timeout_s=900, interval_s=10,
    )

print(json.dumps(list_reservations(brief=True), indent=2))

## 5. Create or reuse the three containers

The SDR path uses a free 10 Gb/s worker interface. The public path uses another free interface. Existing active containers are kept, so re-running the notebook does not destroy the working setup.

In [ ]:
def choose_worker_interfaces(worker_name):
    raw = get_worker_interfaces(worker_name)
    if not raw:
        raise RuntimeError(f"No interface inventory for {worker_name}.")
    interfaces = list(raw.values())[0]
    free_10g, free_other = [], []
    for name, details in interfaces.items():
        if details.get("connections"):
            continue
        (free_10g if str(details.get("speed")) == "10000" else free_other).append(name)
    if not free_10g:
        raise RuntimeError(f"No free 10G interface on {worker_name}: {interfaces}")
    sdr_interface = free_10g.pop(0)
    public_candidates = free_other + free_10g
    if not public_candidates:
        raise RuntimeError(f"No second free interface on {worker_name}: {interfaces}")
    return sdr_interface, public_candidates[0]

def ensure_container(host):
    if get_container_status(host["container"]):
        logger.info(f"Reusing active container {host['container']}.")
        return
    public_ips = get_available_publicips()
    if not public_ips:
        raise RuntimeError("No ExPECA public IP is available.")
    sdr_if, public_if = choose_worker_interfaces(host["worker"])
    chi.container.create_container(
        name=host["container"], image=container_image,
        reservation_id=host["worker_reservation_id"],
        environment={
            "DNS_IP": dns_ip, "GATEWAY_IP": public_gateway,
            "PASS": container_password, "OAI_ROLE": host["role"],
        },
        mounts=[],
        nets=[
            {"network": chi.network.get_network("serverpublic")["id"]},
            {"network": chi.network.get_network(f"{host['sdr']}-sfp-net")["id"]},
        ],
        labels={
            "networks.1.interface": public_if, "networks.1.ip": f"{public_ips[0]}/27",
            "networks.2.interface": sdr_if, "networks.2.ip": host["sdr_host_cidr"],
            "capabilities.privileged": "true",
        },
    )
    chi.container.wait_for_active(host["container"])
    logger.success(f"Created {host['container']} on {host['worker']} with {host['sdr']}.")

for host in radio_hosts:
    ensure_container(host)
    out, _ = remote(host["container"], "ip -br address; uhd_find_devices --args addr=" + host["sdr_device_ip"])
    print(f"\n### {host['role']} / {host['container']}\n{out}")

## 6. Start the 5G Core and gNB

The gNB container hosts the OAI core compose stack. Its gNB configuration is patched only at `sdr_addrs`, preserving the AMF/N2/N3 settings already supplied by the image. The checks require the core services to be healthy and the gNB to associate with the AMF before either UE starts.

In [ ]:
gnb = radio_hosts[0]

core_out, _ = remote(gnb["container"], f"""
cd {shlex.quote(core_root)}
(docker compose up -d || docker-compose up -d)
docker ps --format 'table {{{{.Names}}}}\t{{{{.Status}}}}'
""")
print(core_out)

def core_is_healthy():
    out, rc = remote(gnb["container"], """
bad=$(docker ps --format '{{.Names}} {{.Status}}' | grep '^oai-' | grep -Ev ' Up .*\(healthy\)$| Up [^(]+$' || true)
test -z "$bad" && test $(docker ps --format '{{.Names}}' | grep -c '^oai-') -ge 5
""", check=False)
    return rc == 0

wait_until("OAI core containers are running", core_is_healthy, timeout_s=300, interval_s=10)
for service, expected_ip in core_addresses.items():
    inspect_cmd = "docker inspect -f '{{range .NetworkSettings.Networks}}{{.IPAddress}} {{end}}' oai-" + service
    actual, _ = remote(gnb["container"], inspect_cmd)
    if expected_ip not in actual:
        raise RuntimeError(f"oai-{service} address mismatch: expected {expected_ip}, got {actual}")
    logger.success(f"oai-{service} is reachable at {expected_ip}.")
bridge, _ = remote(gnb["container"], f"ip -4 -o addr show | grep '{gnb_userplane_ip}/26'")
print(bridge)

gnb_cmd = (
    f"./nr-softmodem -O {shlex.quote(gnb_config)} --sa --usrp-tx-thread-config 1 -E "
    "--MACRLCs.[0].dl_max_mcs 20 --MACRLCs.[0].ul_max_mcs 20 "
    "--gNBs.[0].min_rxtxtime 6 --MACRLCs.[0].ulsch_max_frame_inactivity 100"
)
remote(gnb["container"], f"""
sed -i -E 's|sdr_addrs="[^"]*";|sdr_addrs="mgmt_addr={gnb['sdr_device_ip']},addr={gnb['sdr_device_ip']}";|' {shlex.quote(gnb_config)}
grep -E 'amf_ip_address|GNB_IPV4_ADDRESS_FOR_NG_(AMF|U)|sdr_addrs' {shlex.quote(gnb_config)}
grep -q '{core_addresses['amf']}' {shlex.quote(gnb_config)}
grep -q '{gnb_userplane_ip}' {shlex.quote(gnb_config)}
pkill -INT -f '^./nr-softmodem' || true
sleep 5
cd {shlex.quote(oai_build)}
setsid -f {gnb_cmd} </dev/null > /tmp/gnb.log 2>&1
""")

def gnb_ready():
    _, rc = remote(gnb["container"], """
ps -eo args | grep -q '[n]r-softmodem' &&
grep -Eq 'associated AMF|NGSetupResponse|Connected to AMF' /tmp/gnb.log &&
grep -Eq 'ALL RUs ready|RF started|RU.*ready' /tmp/gnb.log
""", check=False)
    return rc == 0

wait_until("gNB RF is running and associated with the AMF", gnb_ready, timeout_s=180)
print(remote(gnb["container"], "tail -80 /tmp/gnb.log")[0])

## 7. Attach UE 2, then UE 1

Before every attempt, the old UE process is interrupted and the notebook waits for UHD to report the SDR as unclaimed. An attempt succeeds only if `nr-uesoftmodem` remains alive and `oaitun_ue1` receives an IPv4 address. The address is discovered dynamically.

In [ ]:
def ue_command(host):
    return (
        "./nr-uesoftmodem --band 77 -C 3919440000 -r 106 --numerology 1 --ssb 396 --sa -E "
        f"--uicc0.imsi {host['imsi']} --uicc0.dnn oai --uicc0.nssai_sst 1 "
        "--uicc0.nssai_sd 16777215 "
        "--uicc0.opc c42449363bbad02b66d16bc975d77cc1 "
        "--uicc0.key fec86ba6eb707ed08905757b1bb44b8f "
        f"--usrp-args \"mgmt_addr={host['sdr_device_ip']},addr={host['sdr_device_ip']}\" "
        f"--ue-fo-compensation --ue-rxgain {ue_rxgain} --ue-txgain {ue_txgain} "
        "--ue-max-power 0 --edaf-addr /tmp/edaf"
    )

def release_ue_sdr(host):
    remote(host["container"], "pkill -INT -f '^./nr-uesoftmodem' || true; sleep 5")
    def is_free():
        out, _ = remote(
            host["container"],
            f"uhd_find_devices --args addr={host['sdr_device_ip']} 2>&1 || true", check=False,
        )
        return "claimed: False" in out
    wait_until(f"{host['sdr']} is unclaimed", is_free, timeout_s=90)

def ue_is_attached(host):
    _, rc = remote(
        host["container"],
        "ps -eo args | grep -q '[n]r-uesoftmodem' && ip -4 addr show dev oaitun_ue1 | grep -q 'inet '",
        check=False,
    )
    return tunnel_cidr(host["container"]) if rc == 0 else None

def attach_ue(host):
    log_path = f"/tmp/{host['role']}.log"
    for attempt in range(1, ue_attach_attempts + 1):
        logger.info(f"Attaching {host['role']} (attempt {attempt}/{ue_attach_attempts})")
        release_ue_sdr(host)
        remote(host["container"], f"""
cd {shlex.quote(oai_build)}
setsid -f {ue_command(host)} </dev/null > {shlex.quote(log_path)} 2>&1
""")
        try:
            cidr = wait_until(
                f"{host['role']} has oaitun_ue1",
                lambda h=host: ue_is_attached(h),
                timeout_s=ue_attach_timeout_s, interval_s=5,
            )
            host["tunnel_cidr"] = cidr
            host["tunnel_ip"] = cidr.split('/')[0]
            return cidr
        except TimeoutError:
            tail, _ = remote(host["container"], f"tail -120 {shlex.quote(log_path)}", check=False)
            print(tail)
    raise RuntimeError(f"{host['role']} did not attach after {ue_attach_attempts} attempts.")

# The verified order is UE 2 first and UE 1 second.
for host in (radio_hosts[2], radio_hosts[1]):
    cidr = attach_ue(host)
    logger.success(f"{host['role']} attached as {cidr}.")
    time.sleep(10)

## 8. Establish and verify end-to-end connectivity

The UE route sends the gNB/core bridge address through the OAI tunnel gateway. A proper setup requires all of the following at once: both UE processes and tunnels are alive, both IMSIs appear as registered in AMF output, the gNB reports two in-sync RNTIs, UPF has both PDU-session addresses, and user-plane pings reach the gNB side.

In [ ]:
ue_hosts = [radio_hosts[1], radio_hosts[2]]
for host in ue_hosts:
    cidr = tunnel_cidr(host["container"])
    if not cidr:
        raise RuntimeError(f"{host['role']} lost oaitun_ue1.")
    host["tunnel_cidr"] = cidr
    host["tunnel_ip"] = cidr.split('/')[0]
    out, _ = remote(host["container"], f"""
ip route replace {gnb_userplane_ip} via {ue_gateway} dev oaitun_ue1
ip -4 addr show dev oaitun_ue1
ip route get {gnb_userplane_ip}
ping -I oaitun_ue1 -c 3 -W 3 {gnb_userplane_ip}
""")
    print(f"\n### {host['role']} user plane\n{out}")

amf, _ = remote(gnb["container"], "docker logs --tail 800 oai-amf 2>&1", check=False)
for host in ue_hosts:
    if host["imsi"] not in amf:
        raise RuntimeError(f"AMF output does not contain {host['imsi']}.")
if amf.count("5GMM-REGISTERED") < 2:
    raise RuntimeError("AMF output does not show two registered UEs.")
print("\n### AMF registration evidence")
print('\n'.join(line for line in amf.splitlines() if "00101000000000" in line or "5GMM-REGISTERED" in line)[-4000:])

gnb_evidence, _ = remote(gnb["container"], "grep -E 'RNTI.*(in[- ]sync|RSRP)|RRCSetupComplete|PDU.*Session' /tmp/gnb.log | tail -120", check=False)
rntis = set(re.findall(r"RNTI\s+(?:0x)?([0-9a-fA-F]+)", gnb_evidence))
if len(rntis) < 2:
    raise RuntimeError(f"Expected two gNB RNTIs; found {sorted(rntis)}.\n{gnb_evidence}")
print(f"\n### gNB evidence ({len(rntis)} RNTIs)\n{gnb_evidence}")

upf, _ = remote(gnb["container"], "docker logs --tail 800 oai-upf 2>&1", check=False)
missing_upf = [h["tunnel_ip"] for h in ue_hosts if h["tunnel_ip"] not in upf]
if missing_upf:
    logger.warning(f"UPF log format did not expose these active tunnel IPs: {missing_upf}")
print("\n### UPF PDR/FAR evidence")
print('\n'.join(line for line in upf.splitlines() if any(ip in line for ip in [h["tunnel_ip"] for h in ue_hosts]) or "PDR" in line or "FAR" in line)[-5000:])

logger.success("Both UEs are attached, registered, routed, and passing user-plane traffic.")

## 9. D-ITG end-to-end smoke measurement

The gNB container receives UDP traffic at `192.168.70.129`. Each UE sends 100-byte packets with a 50 ms inter-departure time (about 20 packets/s). The receiver logs are decoded on the gNB. This is a short health test; extend `ditg_duration_ms` for benchmark runs.

In [ ]:
ditg_repo = "https://github.com/gourav-prateek-sharma/ditg.git"
for host in radio_hosts:
    remote(host["container"], f"test -d /root/ditg/.git || git clone {shlex.quote(ditg_repo)} /root/ditg")

remote(gnb["container"], """
pkill -INT -f '^./ITGRecv' || true
sleep 2
cd /root/ditg/src/ITGRecv
setsid -f ./ITGRecv </dev/null > /tmp/itg-recv.log 2>&1
""")
time.sleep(3)

ditg_results = {}
for host in ue_hosts:
    receiver_log = f"{host['role']}_receiver.log"
    sender_log = f"/tmp/{host['role']}_sender.log"
    send_out, _ = remote(host["container"], f"""
cd /root/ditg/src/ITGSend
printf '{ditg_idt_ms}\n' > idt.txt
./ITGSend -T UDP -a {gnb_userplane_ip} -c {ditg_packet_bytes} -Ft idt.txt -t {ditg_duration_ms} -l {sender_log} -x {receiver_log}
""")
    time.sleep(2)
    decoded, _ = remote(gnb["container"], f"cd /root/ditg/src/ITGDec && ./ITGDec ../ITGRecv/{receiver_log}")
    ditg_results[host["role"]] = decoded
    print(f"\n### {host['role']} D-ITG\n{send_out}\n{decoded}")

logger.success("D-ITG smoke measurements completed for both UEs.")

## 10. Collect EDAF and traffic artifacts

EDAF writes timestamped `/tmp/edaf*.lseq` files (not a directory). D-ITG receiver logs remain under `/root/ditg/src/ITGRecv`. Copy them out before a container is deleted.

In [ ]:
for host in radio_hosts:
    artifacts, _ = remote(host["container"], """
find /tmp -maxdepth 1 -type f -name 'edaf*.lseq' -printf '%TY-%Tm-%Td %TH:%TM:%TS %p\n' 2>/dev/null | sort
find /root/ditg -type f \( -name '*.log' -o -name '*.json.gz' \) -printf '%p\n' 2>/dev/null | sort | tail -40
""", check=False)
    print(f"\n### {host['role']} artifacts\n{artifacts or '(none yet)'}")

## Verified reference result (2026-09-30)

The bring-up used this exact topology. UE 2 received `10.0.0.3/24`; UE 1 received `10.0.0.5/24`; both appeared `5GMM-REGISTERED`; the gNB reported two in-sync RNTIs; and UPF installed PDR/FAR state for both PDU sessions. A 10-second D-ITG smoke run produced 0% loss and 19.479 ms mean delay for UE 2, and 3.48% loss and 84.343 ms mean delay for UE 1. These are health-check observations, not benchmark conclusions. Tunnel addresses may differ on a later run.

## Optional: stop radio and traffic processes

This cell stops only the processes started above. It does not stop the core, delete containers, or release leases. Set `stop_processes = True` deliberately before running it.

In [ ]:
stop_processes = False
if stop_processes:
    for host in radio_hosts[1:]:
        remote(host["container"], "pkill -INT -f '^./nr-uesoftmodem' || true")
    remote(gnb["container"], "pkill -INT -f '^./ITGRecv' || true; pkill -INT -f '^./nr-softmodem' || true")
    logger.success("Stopped UE, gNB, and D-ITG receiver processes; core and resources remain allocated.")
else:
    print("No action taken.")